## 0. Configurando sessão spark

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_aluno")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

# Projeto usado para faturamento das consultas
spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [10]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [11]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [12]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_silver_aluno = f"{par_source_project}.silver.aluno"

par_source_gold_fato_aluno = f"{par_source_project}.gold.fato_indicador_aluno"

## 3. Leitura dos dados da origem

In [13]:
df_scr_aluno = spark.read.format("bigquery").option("table",par_source_silver_aluno).load()

## 4. Transformações

In [14]:
fato_aluno = (
    df_scr_aluno
    .select("ano","id_municipio","id_escola","id_aluno","caderno","serie","rede_id",
            "proficiencia","peso_aluno",
            "presenca","preenchimento_caderno","alfabetizado",
            "presenca_id","preenchimento_caderno_id","alfabetizado_id")
)

## 5. Armazenamento no BQ

In [16]:
(
    fato_aluno.write.format("bigquery")
    .option("table", par_source_gold_fato_aluno)
    .option("writeMethod", "direct")
    .mode("overwrite")
    .save()
)

26/08/26 01:31:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                